DELIVERABLE 2
Student(s): Hannan And Maja
INF201
This notebook processes coffe shpt transaction logs, cleans up the data, performs analysisr, and generates reports.
Repository link: 

In [69]:
# Environment check

from pathlib import Path

for item in Path("d2_data").rglob("*"):
    print(item)

d2_data\2026
d2_data\store_config.json
d2_data\2026\library_cafe
d2_data\2026\science_building
d2_data\2026\student_union
d2_data\2026\library_cafe\POS_01
d2_data\2026\library_cafe\POS_02
d2_data\2026\library_cafe\POS_03
d2_data\2026\library_cafe\POS_01\register_sales.log
d2_data\2026\library_cafe\POS_02\register_sales.log
d2_data\2026\library_cafe\POS_03\register_sales.log
d2_data\2026\science_building\POS_01
d2_data\2026\science_building\POS_02
d2_data\2026\science_building\POS_03
d2_data\2026\science_building\POS_01\register_sales.log
d2_data\2026\science_building\POS_02\register_sales.log
d2_data\2026\science_building\POS_03\register_sales.log
d2_data\2026\student_union\POS_01
d2_data\2026\student_union\POS_02
d2_data\2026\student_union\POS_03
d2_data\2026\student_union\POS_01\register_sales.log
d2_data\2026\student_union\POS_02\register_sales.log
d2_data\2026\student_union\POS_03\register_sales.log


In [70]:
from pathlib import Path
import json
import re
import pandas as pd
import numpy as np


In [71]:
root = Path("d2_data")

for file in root.rglob("*"):
    print(file)
    

d2_data\2026
d2_data\store_config.json
d2_data\2026\library_cafe
d2_data\2026\science_building
d2_data\2026\student_union
d2_data\2026\library_cafe\POS_01
d2_data\2026\library_cafe\POS_02
d2_data\2026\library_cafe\POS_03
d2_data\2026\library_cafe\POS_01\register_sales.log
d2_data\2026\library_cafe\POS_02\register_sales.log
d2_data\2026\library_cafe\POS_03\register_sales.log
d2_data\2026\science_building\POS_01
d2_data\2026\science_building\POS_02
d2_data\2026\science_building\POS_03
d2_data\2026\science_building\POS_01\register_sales.log
d2_data\2026\science_building\POS_02\register_sales.log
d2_data\2026\science_building\POS_03\register_sales.log
d2_data\2026\student_union\POS_01
d2_data\2026\student_union\POS_02
d2_data\2026\student_union\POS_03
d2_data\2026\student_union\POS_01\register_sales.log
d2_data\2026\student_union\POS_02\register_sales.log
d2_data\2026\student_union\POS_03\register_sales.log


TASK 1 / READING AND PARSING LOG FILES
This section contains functions for:
Reading log files and unknown encoding
Parsing transation records using regular expressions
Ignoring non-transation log entries

In [72]:
from pathlib import Path

def read_log_file(file_path):
    """
    Reads a log file and returns a list of dictionaries containing the log entries.
    
    Args:
        file_path (str): The path to the log file.      
    """
    file_path = Path(file_path)

    try:
        text = file_path.read_text(encoding='utf-8')
        return text, "utf-8"

    except UnicodeDecodeError:
        text = file_path.read_text(
            encoding="iso-8859-1"
        )
        return text, "iso-8859-1"
    



In [73]:
sample_file =Path(
"d2_data/2026/library_cafe/POS_01/register_sales.log"        
)

content, encoding = read_log_file(sample_file)

print("Encoding:", encoding)
print(content[:500])  # Print the first 500 characters of the content


Encoding: utf-8
[2026-10-15 07:02:33] [POS_01] item=Espresso price=2.89 USD (REFUND)
[2026-10-15 07:03:38] [POS_01] item=Latte price=4.25 USD (SUCCESS)
[2026-10-15 07:05:02] [POS_01] item=Espresso price=3.38 USD (SUCCESS)
=== NETWORK SYNC OK ===
[2026-10-15 07:07:40] [POS_01] item=Sandwich price=7.21 EUR (SUCCESS)
--- PRINTER PAPER LOW ---
[2026-10-15 07:07:51] [POS_01] item=Latte price=4.87 EUR (SUCCESS)
[2026-10-15 07:09:53] [POS_01] item=Sandwich price=7.77 NOK (SUCCESS)
[DEBUG] BARCODE SCANNER REBOOTED
[202


In [74]:
import re

In [75]:
# Extracts transaction information from valid log entries.
# Non-transaction noise lines are ignored.

pattern = re.compile(
    r"\[(.*?)\]\s"
    r"\[(.*?)\]\s"
    r"item=(.*?)\s"
    r"price=([\d.]+)\s"
    r"(\w+)\s"
    r"\((.*?)\)"
)

In [76]:
line = "[2026-10-15 07:02:13] [POS_01] item=Latte price=4.45 EUR (SUCCESS)"

In [77]:
match = pattern.match(line)

print(match.groups())

('2026-10-15 07:02:13', 'POS_01', 'Latte', '4.45', 'EUR', 'SUCCESS')


In [78]:
(
 '2026-10-15 07:02:13',
 'POS_01',
 'Latte',
 '4.45',
 'EUR',
 'SUCCESS'
)

('2026-10-15 07:02:13', 'POS_01', 'Latte', '4.45', 'EUR', 'SUCCESS')

In [79]:
def parse_log(text):

    records = []

    for line in text.splitlines():

        match = pattern.match(line)

        if match:

            record = {
                "timestamp": match.group(1),
                "register_id": match.group(2),
                "item": match.group(3),
                "price": float(match.group(4)),
                "currency": match.group(5),
                "status": match.group(6)
            }

            records.append(record)

    return records

In [80]:
#Teast parse on a sample log file.

sample_file = Path(
    "d2_data/2026/library_cafe/POS_01/register_sales.log"
)

content, encoding = read_log_file(sample_file)

records = parse_log(content)

In [81]:
print(len(records))

80


In [82]:
records[:3]

[{'timestamp': '2026-10-15 07:02:33',
  'register_id': 'POS_01',
  'item': 'Espresso',
  'price': 2.89,
  'currency': 'USD',
  'status': 'REFUND'},
 {'timestamp': '2026-10-15 07:03:38',
  'register_id': 'POS_01',
  'item': 'Latte',
  'price': 4.25,
  'currency': 'USD',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:05:02',
  'register_id': 'POS_01',
  'item': 'Espresso',
  'price': 3.38,
  'currency': 'USD',
  'status': 'SUCCESS'}]

In [83]:
all_records = []

for log_file in Path("d2_data").rglob("*.log"):

    content, encoding = read_log_file(log_file)

    records = parse_log(content)

    print(log_file.name,
          encoding,
          len(records))

    all_records.extend(records)

register_sales.log utf-8 80
register_sales.log utf-8 80
register_sales.log utf-8 80
register_sales.log utf-8 80
register_sales.log utf-8 80
register_sales.log utf-8 80
register_sales.log utf-8 80
register_sales.log utf-8 80
register_sales.log utf-8 80


In [84]:
len(all_records)

720

In [85]:
all_records[:3]

[{'timestamp': '2026-10-15 07:02:33',
  'register_id': 'POS_01',
  'item': 'Espresso',
  'price': 2.89,
  'currency': 'USD',
  'status': 'REFUND'},
 {'timestamp': '2026-10-15 07:03:38',
  'register_id': 'POS_01',
  'item': 'Latte',
  'price': 4.25,
  'currency': 'USD',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:05:02',
  'register_id': 'POS_01',
  'item': 'Espresso',
  'price': 3.38,
  'currency': 'USD',
  'status': 'SUCCESS'}]

In [86]:
len(all_records)

all_records[:5]


[{'timestamp': '2026-10-15 07:02:33',
  'register_id': 'POS_01',
  'item': 'Espresso',
  'price': 2.89,
  'currency': 'USD',
  'status': 'REFUND'},
 {'timestamp': '2026-10-15 07:03:38',
  'register_id': 'POS_01',
  'item': 'Latte',
  'price': 4.25,
  'currency': 'USD',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:05:02',
  'register_id': 'POS_01',
  'item': 'Espresso',
  'price': 3.38,
  'currency': 'USD',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:07:40',
  'register_id': 'POS_01',
  'item': 'Sandwich',
  'price': 7.21,
  'currency': 'EUR',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:07:51',
  'register_id': 'POS_01',
  'item': 'Latte',
  'price': 4.87,
  'currency': 'EUR',
  'status': 'SUCCESS'}]

In [87]:
print(len(records))

80


In [88]:
all_records = []

for log_file in Path("d2_data").rglob("*.log"):

    content, encoding = read_log_file(log_file)

    records = parse_log(content)

    print(log_file.name,
          encoding,
          len(records))

    all_records.extend(records)

print("\nTotal records:", len(all_records))

register_sales.log utf-8 80
register_sales.log utf-8 80
register_sales.log utf-8 80
register_sales.log utf-8 80
register_sales.log utf-8 80
register_sales.log utf-8 80
register_sales.log utf-8 80
register_sales.log utf-8 80
register_sales.log utf-8 80

Total records: 720


In [89]:
all_records[0]

{'timestamp': '2026-10-15 07:02:33',
 'register_id': 'POS_01',
 'item': 'Espresso',
 'price': 2.89,
 'currency': 'USD',
 'status': 'REFUND'}

TASK 2: RECURSIVE LOG SEARCH
Recursive log search
Recursive JSON search.

In [90]:
# Recusrsively transverse folder and collects all .log files.

from pathlib import Path

def find_log_files(folder):

    folder = Path(folder)

    log_files = []

    for item in folder.iterdir():

        if item.is_dir():

            log_files.extend(
                find_log_files(item)
            )

        elif item.is_file() and item.suffix == ".log":

            log_files.append(item)

    return log_files


In [91]:
log_files = find_log_files("d2_data")

print(len(log_files))

for file in log_files:
    print(file)


9
d2_data\2026\library_cafe\POS_01\register_sales.log
d2_data\2026\library_cafe\POS_02\register_sales.log
d2_data\2026\library_cafe\POS_03\register_sales.log
d2_data\2026\science_building\POS_01\register_sales.log
d2_data\2026\science_building\POS_02\register_sales.log
d2_data\2026\science_building\POS_03\register_sales.log
d2_data\2026\student_union\POS_01\register_sales.log
d2_data\2026\student_union\POS_02\register_sales.log
d2_data\2026\student_union\POS_03\register_sales.log


In [92]:
import json

with open(
    "d2_data/store_config.json",
    "r",
    encoding="utf-8"
) as f:

    config = json.load(f)
    

In [93]:
# Recusrively seaches nested dictionaries and lists
# to identify active register IDs.

def get_active_registers(data):

    active = set()

    if isinstance(data, dict):

        if (
            "id" in data
            and data.get("active") is True
        ):
            active.add(data["id"])

        for value in data.values():
            active.update(
                get_active_registers(value)
            )

    elif isinstance(data, list):

        for item in data:
            active.update(
                get_active_registers(item)
            )

    return active

In [94]:
active_registers = get_active_registers(config)

print(active_registers)

{'POS_01', 'POS_02', 'POS_03'}


In [95]:
print(len(log_files))
print(active_registers)

9
{'POS_01', 'POS_02', 'POS_03'}


In [96]:
all_records = []

for file in log_files:
    
    content, encoding = read_log_file(file)
    
    records = parse_log(content)

    all_records.extend(records)

print(len(all_records))


720


In [97]:
prices = [record["price"] for record in all_records]


TASK 3: DATA CLEANING.
Outlier Removal
Refund Ratio.

In [98]:
# Detects price outliers using the 3-sigma rule and
# replaces exteme values with the median.

import numpy as np

def remove_outliers(prices):

    prices = np.array(prices, dtype=float)

    mean = np.mean(prices)
    std = np.std(prices)

    outlier_mask = np.abs(
        prices - mean
    ) > 3 * std

    valid_prices = prices[~outlier_mask]

    median = np.median(valid_prices)

    prices[outlier_mask] = median

    return prices


In [99]:
cleaned_prices = remove_outliers(prices)

print(max(prices))
print(max(cleaned_prices))


999.0
8.0


In [100]:
prices = np.array(prices)

mean = np.mean(prices)
std = np.std(prices)

outliers = np.abs(
    prices - mean
) > 3 * std

print("Outliers:", outliers.sum())


Outliers: 15


In [101]:
# Computes refund ratio while avoiding division-by-zero errors.


def compute_refund_ratio(
    sales_count,
    refund_count
):

    return refund_count / (
        sales_count + 1e-6
    )


In [102]:
compute_refund_ratio(100, 5)


0.0499999995

In [103]:
prices = [r["price"] for r in all_records]

cleaned_prices = remove_outliers(prices)

print("Original Max:", max(prices))
print("Cleaned Max:", max(cleaned_prices))


Original Max: 999.0
Cleaned Max: 8.0


In [104]:
def compute_refund_ratio(
    sales_count,
    refund_count
):

    return refund_count / (
        sales_count + 1e-6
    )

In [105]:
print(
    compute_refund_ratio(
        sales_count=100,
        refund_count=5
    )
)

0.0499999995


TASK 4: DATA ANALYSIS
Build Dataframe.

In [106]:
# Converts parsed transaction records into panda DataFrame.
# Timestamps are converted to datetime format and used as index.

df = pd.DataFrame(all_records)

In [107]:
df.head()


,timestamp,register_id,item,price,currency,status
0,2026-10-15 07:02:33,POS_01,Espresso,2.89,USD,REFUND
1,2026-10-15 07:03:38,POS_01,Latte,4.25,USD,SUCCESS
2,2026-10-15 07:05:02,POS_01,Espresso,3.38,USD,SUCCESS
3,2026-10-15 07:07:40,POS_01,Sandwich,7.21,EUR,SUCCESS
4,2026-10-15 07:07:51,POS_01,Latte,4.87,EUR,SUCCESS


In [108]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

In [109]:
df = df.sort_values("timestamp")

In [110]:
df = df.set_index("timestamp")

In [111]:
df.head()


,register_id,item,price,currency,status
timestamp,,,,,
2026-10-15 07:00:38,POS_01,Muffin,4.00,EUR,SUCCESS
2026-10-15 07:00:46,POS_03,Latte,4.78,USD,CANCELLED
2026-10-15 07:01:18,POS_02,Latte,4.67,NOK,SUCCESS
2026-10-15 07:01:36,POS_01,Croissant,4.11,NOK,REFUND
2026-10-15 07:02:01,POS_01,Cappuccino,4.20,USD,SUCCESS


In [112]:
# Assignment requirement:
# NOK transations must be multiplied by 10.

df.loc[df["currency"] == "NOK", "price"] *= 10.0

In [113]:
df.head()

,register_id,item,price,currency,status
timestamp,,,,,
2026-10-15 07:00:38,POS_01,Muffin,4.00,EUR,SUCCESS
2026-10-15 07:00:46,POS_03,Latte,4.78,USD,CANCELLED
2026-10-15 07:01:18,POS_02,Latte,46.70,NOK,SUCCESS
2026-10-15 07:01:36,POS_01,Croissant,41.10,NOK,REFUND
2026-10-15 07:02:01,POS_01,Cappuccino,4.20,USD,SUCCESS


In [114]:
# Generates revenus totals grouped by item and hour.

def hourly_revenue(df):

    revenue = (
        df.groupby("item")
          .resample("1h")
          ["price"]
          .sum()
    )

    return revenue

In [115]:
hourly_report = hourly_revenue(df)

hourly_report.head()

item        timestamp          
Cappuccino  2026-10-15 07:00:00    1767.41
            2026-10-15 08:00:00    2781.34
            2026-10-15 09:00:00     106.40
Croissant   2026-10-15 07:00:00    2020.21
            2026-10-15 08:00:00     767.60
Name: price, dtype: float64

In [116]:
# Summarises transcation values by item and transation status.

def status_pivot(df):

    return pd.pivot_table(
        df,
        values="price",
        index="item",
        columns="status",
        aggfunc="sum",
        fill_value=0
    )

In [117]:
pivot_table = status_pivot(df)

pivot_table


status,CANCELLED,REFUND,SUCCESS
item,,,
Cappuccino,1131.18,1282.79,2241.18
Croissant,451.10,1277.25,1165.33
Espresso,132.10,1140.80,13036.90
Latte,10513.46,312.42,2184.80
Muffin,200.95,10289.83,3104.41
Sandwich,611.49,529.31,4627.89


In [118]:
# Calculates the refund ration for each time.

def refund_ratio_per_item(df):

    sales = (
        df[df["status"] == "SUCCESS"]
        .groupby("item")
        .size()
    )

    refunds = (
        df[df["status"] == "REFUND"]
        .groupby("item")
        .size()
    )

    sales, refunds = sales.align(
        refunds,
        fill_value=0
    )

    ratios = compute_refund_ratio(
        sales.to_numpy(),
        refunds.to_numpy()
    )

    return pd.DataFrame({
        "item": sales.index,
        "refund_ratio": ratios
    })

In [119]:
refund_report = refund_ratio_per_item(df)

refund_report


,item,refund_ratio
0,Cappuccino,0.205128
1,Croissant,0.225000
2,Espresso,0.276316
3,Latte,0.164557
4,Muffin,0.170213
5,Sandwich,0.207317


In [120]:
df.to_csv("cleaned_transactions.csv")

In [121]:
hourly_report.to_json("hourly_revenue.json")


In [122]:
pivot_table.to_csv("status_pivot_table.csv")


In [123]:
refund_report.to_csv(
    "item_refund_ratios.csv",
    index=False
)


TASK 5: COMPLETE WORKFLOW

In [124]:
# Executes the full analysis pipeline:
# 1. Locate log files
# 2. Load configuration
# 3. Extract active registers
# 4. Parse transations
# 5. Build DataFrame
# 6. Clean data
# 7. Generate reports
# 8. Save outputs

def main():

    # Find log files
    log_files = find_log_files("d2_data")

    # Load config
    with open(
        "d2_data/store_config.json",
        "r",
        encoding="utf-8"
    ) as f:

        config = json.load(f)

    # Active registers
    active_registers = get_active_registers(config)

    print("Active Registers:")
    print(active_registers)

    # Read and parse logs
    all_records = []

    for file in log_files:

        content, encoding = read_log_file(file)

        records = parse_log(content)

        all_records.extend(records)

    print(
        f"Parsed {len(all_records)} records"
    )

    # Build DataFrame
    df = build_dataframe(all_records)

    # Remove outliers
    cleaned_prices = remove_outliers(
        df["price"]
    )

    df["price"] = cleaned_prices

    # Correct NOK prices
    df.loc[
        df["currency"] == "NOK",
        "price"
    ] *= 10

    # Generate reports
    hourly_report = hourly_revenue(df)

    pivot_table = status_pivot(df)

    refund_report = refund_ratio_per_item(df)

    # Save outputs
    df.to_csv(
        "cleaned_transactions.csv"
    )

    hourly_report.to_json(
        "hourly_revenue.json"
    )

    pivot_table.to_csv(
        "status_pivot_table.csv"
    )

    refund_report.to_csv(
        "item_refund_ratios.csv",
        index=False
    )

    print("Files created successfully!")

    main()